# Saúde Mental no Trabalho Tech
## Análise Narrativa em 4 Atos — OSMI Mental Health in Tech Survey

**Objetivo:** Provar, com dados, que o problema da saúde mental nas empresas de tech
começa antes da contratação, é amplificado pelo ambiente corporativo,
falha por causa da cultura do medo e pode ser antecipado por um modelo preditivo de ML.

**Dataset:** OSMI Mental Health in Tech Survey — 1.251 respondentes, 24 colunas.


## Seção 0 — Setup e Carregamento

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 13, 'axes.labelsize': 11})

CSV_PATH = "data/processed/survey_limpo.csv"
df = pd.read_csv(CSV_PATH)

print(f"Shape: {df.shape}")
print(f"Colunas: {list(df.columns)}")
df.head(3)


In [ ]:
# Visão geral do dataset
print("=== TIPOS E NULOS ===")
print(df.info())
print()
print("=== NULOS POR COLUNA ===")
print(df.isnull().sum()[df.isnull().sum() > 0])


In [ ]:
# Distribuição das colunas principais do storytelling
colunas_chave = [
    'family_history', 'treatment', 'mental_health_interview', 'phys_health_interview',
    'remote_work', 'work_interfere', 'coworkers', 'supervisor',
    'no_employees', 'leave', 'anonymity', 'obs_consequence'
]
for col in colunas_chave:
    print(f"\n{col}:")
    print(df[col].value_counts())


---
## Ato 1 — A Bagagem Invisível (Antes do Primeiro Dia)

**Premissa:** O problema não começa quando o funcionário é contratado.
Ele já chega carregando uma história — e, por medo, não conta ao RH.

O RH contrata **no escuro**.


### 1.1 — A Carga Pré-existente: Histórico Familiar × Busca por Tratamento

In [ ]:
df_fam = df[df['family_history'].isin(['Yes', 'No']) & df['treatment'].isin(['Yes', 'No'])].copy()

tab = pd.crosstab(df_fam['family_history'], df_fam['treatment'])
tab_pct = tab.div(tab.sum(axis=1), axis=0) * 100

print("Proporção de busca por tratamento por histórico familiar:")
print(tab_pct.round(1))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absoluto
tab.plot(kind='bar', ax=axes[0], color=['#4db6ac', '#e57373'], edgecolor='none', rot=0)
axes[0].set_title('Histórico Familiar × Tratamento (Absoluto)')
axes[0].set_xlabel('Histórico Familiar')
axes[0].set_ylabel('Respondentes')
axes[0].legend(title='Buscou Tratamento?', frameon=False)
for c in axes[0].containers:
    axes[0].bar_label(c, padding=3, fontsize=9)

# Percentual
tab_pct.plot(kind='bar', ax=axes[1], color=['#4db6ac', '#e57373'], edgecolor='none', rot=0)
axes[1].set_title('Histórico Familiar × Tratamento (%)')
axes[1].set_xlabel('Histórico Familiar')
axes[1].set_ylabel('Proporção (%)')
axes[1].legend(title='Buscou Tratamento?', frameon=False)
for c in axes[1].containers:
    axes[1].bar_label(c, fmt='%.1f%%', padding=3, fontsize=9)

plt.tight_layout()
plt.savefig('data/processed/ato1_historico_tratamento.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Funcionários com histórico familiar de doenças mentais buscam tratamento em
proporção significativamente maior. A bagagem invisível é measurável e começa antes da contratação.


### 1.2 — A Omissão na Entrevista: Mental × Físico

In [ ]:
df_ent = df[
    df['mental_health_interview'].isin(['Yes', 'No', 'Maybe']) &
    df['phys_health_interview'].isin(['Yes', 'No', 'Maybe'])
].copy()

mental_pct = df_ent['mental_health_interview'].value_counts(normalize=True) * 100
fisico_pct = df_ent['phys_health_interview'].value_counts(normalize=True) * 100

print("Disposição de revelar na entrevista (%):")
comp = pd.DataFrame({'Saúde Mental': mental_pct, 'Saúde Física': fisico_pct}).round(1)
print(comp)

fig, ax = plt.subplots(figsize=(10, 5))

x = np.arange(3)
labels = ['Yes', 'Maybe', 'No']
w = 0.35

bars1 = ax.bar(x - w/2,
               [mental_pct.get(l, 0) for l in labels],
               w, label='Saúde Mental', color='#e57373', edgecolor='none')
bars2 = ax.bar(x + w/2,
               [fisico_pct.get(l, 0) for l in labels],
               w, label='Saúde Física', color='#4db6ac', edgecolor='none')

ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_title('Disposição de Revelar Condições de Saúde na Entrevista (%)')
ax.set_xlabel('Resposta')
ax.set_ylabel('Proporção (%)')
ax.legend(frameon=False)
ax.bar_label(bars1, fmt='%.1f%%', padding=3, fontsize=9)
ax.bar_label(bars2, fmt='%.1f%%', padding=3, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/ato1_entrevista.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Candidatos se sentem muito mais confortáveis revelando problemas físicos do que mentais.
A diferença nas respostas 'Yes' é expressiva — candidatos omitem sua condição mental para não serem discriminados.

**GANCHO:** *"Se os funcionários já chegam com problemas e têm medo de falar sobre eles
na entrevista, o RH contrata no escuro — sem saber a realidade de quem está contratando."*


---
## Ato 2 — O Ecossistema Corporativo (Como o Ambiente Molda o Funcionário)

**Premissa:** Uma vez dentro da empresa, como a rotina afeta a mente?
O regime de trabalho e o tamanho da empresa fazem diferença real.


### 2.1 — O Paradoxo do Trabalho Remoto

In [ ]:
df_rem = df[
    df['remote_work'].isin(['Yes', 'No']) &
    df['work_interfere'].isin(['Often', 'Sometimes', 'Rarely', 'Never'])
].copy()

ORDEM_WI = ['Never', 'Rarely', 'Sometimes', 'Often']

tab_rem = pd.crosstab(df_rem['remote_work'], df_rem['work_interfere'])
tab_rem = tab_rem[ORDEM_WI]
tab_rem_pct = (tab_rem.div(tab_rem.sum(axis=1), axis=0) * 100).round(1)

print("Interferência no trabalho por regime (%):")
print(tab_rem_pct)

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(ORDEM_WI))
w = 0.35
cores_wi = ['#4db6ac', '#80cbc4', '#ffb74d', '#e57373']

for i, (regime, label) in enumerate([('Yes', 'Remoto'), ('No', 'Presencial')]):
    vals = [tab_rem_pct.loc[regime, cat] for cat in ORDEM_WI]
    offset = (i - 0.5) * w
    bars = ax.bar(x + offset, vals, w, label=label, color=cores_wi if i == 0 else ['#b2dfdb','#e0f2f1','#ffe0b2','#ffcdd2'],
                  edgecolor='none')
    ax.bar_label(bars, fmt='%.1f%%', padding=3, fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(ORDEM_WI)
ax.set_title('Interferência da Saúde Mental × Regime de Trabalho (%)')
ax.set_xlabel('Nível de Interferência')
ax.set_ylabel('Proporção (%)')
ax.legend(title='Regime', frameon=False)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/ato2_remoto_interferencia.png', bbox_inches='tight')
plt.show()


In [ ]:
# Suporte social por regime: supervisor e colegas
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, col, titulo in zip(
    axes,
    ['supervisor', 'coworkers'],
    ['Conforto em Falar com Supervisor', 'Conforto em Falar com Colegas']
):
    df_s = df[df['remote_work'].isin(['Yes', 'No']) & df[col].isin(['Yes', 'No', 'Some of them'])].copy()
    tab = pd.crosstab(df_s['remote_work'], df_s[col], normalize='index') * 100
    tab = tab.rename(index={'Yes': 'Remoto', 'No': 'Presencial'})
    cols_ordem = [c for c in ['Yes', 'Some of them', 'No'] if c in tab.columns]
    tab[cols_ordem].plot(kind='bar', ax=ax, color=['#4db6ac', '#ffb74d', '#e57373'],
                          edgecolor='none', rot=0)
    ax.set_title(f'{titulo} × Regime (%)')
    ax.set_xlabel('')
    ax.set_ylabel('Proporção (%)')
    ax.legend(frameon=False, fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    for c in ax.containers:
        ax.bar_label(c, fmt='%.1f%%', padding=2, fontsize=8)

plt.tight_layout()
plt.savefig('data/processed/ato2_suporte_social.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Profissionais remotos relatam maior interferência 'Often' e 'Sometimes'
e, ao mesmo tempo, têm MENOS facilidade de conversar com supervisores e colegas.
O isolamento do trabalho remoto amplifica o problema sem oferecer rede de suporte.


### 2.2 — O Tamanho da Empresa e o Acesso à Licença

In [ ]:
mapa_empresa = {
    '1-5':            'Startup (1-5)',
    '6-25':           'Startup (6-25)',
    '26-100':         'Pequena (26-100)',
    '100-500':        'Média (100-500)',
    '500-1000':       'Grande (500-1000)',
    'More than 1000': 'Corporação (1000+)',
}
ordem_empresa = list(mapa_empresa.values())
ordem_leave   = ['Very easy', 'Somewhat easy', "Don't know", 'Somewhat difficult', 'Very difficult']

df_emp = df[
    df['no_employees'].isin(mapa_empresa.keys()) &
    df['leave'].isin(ordem_leave)
].copy()
df_emp['Empresa'] = df_emp['no_employees'].map(mapa_empresa)

tab_emp = pd.crosstab(df_emp['Empresa'], df_emp['leave'])
tab_emp = tab_emp[[c for c in ordem_leave if c in tab_emp.columns]]
tab_emp_pct = (tab_emp.div(tab_emp.sum(axis=1), axis=0) * 100).round(1)
tab_emp_pct = tab_emp_pct.reindex([e for e in ordem_empresa if e in tab_emp_pct.index])

print("Facilidade de licença por tamanho de empresa (%):")
print(tab_emp_pct)

fig, ax = plt.subplots(figsize=(13, 6))
tab_emp_pct.plot(kind='bar', ax=ax,
                 color=['#4db6ac', '#80cbc4', '#90a4ae', '#ffb74d', '#e57373'],
                 edgecolor='none', rot=30)
ax.set_title('Facilidade de Tirar Licença Médica × Tamanho da Empresa (%)')
ax.set_xlabel('Porte da Empresa')
ax.set_ylabel('Proporção (%)')
ax.legend(title='Facilidade de Licença', frameon=False, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/ato2_empresa_licenca.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Startups menores tendem a ter processos mais informais para licença médica.
Grandes corporações concentram mais 'Don't know', indicando que o processo existe mas não é comunizado —
os funcionários não sabem como acessá-lo.


---
## Ato 3 — A Falha do Sistema e a Cultura do Medo (O Clímax)

**Premissa:** Não basta a empresa oferecer benefícios se a cultura for punitiva.
O sigilo não garantido e o medo aprendido são as verdadeiras barreiras.


### 3.1 — A Barreira do Anonimato

In [ ]:
df_anon = df[
    df['anonymity'].isin(['Yes', 'No', "Don't know"]) &
    df['treatment'].isin(['Yes', 'No'])
].copy()

tab_anon = pd.crosstab(df_anon['anonymity'], df_anon['treatment'])
tab_anon_pct = (tab_anon.div(tab_anon.sum(axis=1), axis=0) * 100).round(1)

print("Busca por tratamento × garantia de anonimato (%):")
print(tab_anon_pct)

mapa_anon = {'Yes': 'Sigilo Garantido', 'No': 'Sem Sigilo', "Don't know": 'Não Sabe'}
tab_plot = tab_anon_pct.rename(index=mapa_anon)

fig, ax = plt.subplots(figsize=(9, 5))
tab_plot[['Yes', 'No']].plot(
    kind='bar', ax=ax, color=['#4db6ac', '#e57373'], edgecolor='none', rot=0
)
ax.set_title('Garantia de Anonimato × Busca por Tratamento (%)')
ax.set_xlabel('Anonimato Garantido pela Empresa')
ax.set_ylabel('Proporção (%)')
ax.legend(title='Buscou Tratamento?', frameon=False)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', padding=3, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/ato3_anonimato.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Com sigilo garantido, a taxa de busca por tratamento é significativamente maior.
Sem sigilo, ela despenca. Quando a empresa não comunica claramente ('Don't know'), o efeito é intermediário.

**Conclusão:** O anonimato não é um detalhe — é condição necessária para que qualquer programa funcione.


### 3.2 — A Cultura da Punição

In [ ]:
df_pun = df[
    df['obs_consequence'].isin(['Yes', 'No']) &
    df['treatment'].isin(['Yes', 'No'])
].copy()

tab_pun = pd.crosstab(df_pun['obs_consequence'], df_pun['treatment'])
tab_pun_pct = (tab_pun.div(tab_pun.sum(axis=1), axis=0) * 100).round(1)

print("Busca por tratamento × observação de consequências negativas (%):")
print(tab_pun_pct)

mapa_pun = {'Yes': 'Viu Colegas Punidos', 'No': 'Não Viu Punições'}
tab_plot_pun = tab_pun_pct.rename(index=mapa_pun)

fig, ax = plt.subplots(figsize=(9, 5))
tab_plot_pun[['Yes', 'No']].plot(
    kind='bar', ax=ax, color=['#4db6ac', '#e57373'], edgecolor='none', rot=0
)
ax.set_title('Observação de Consequências Negativas × Busca por Tratamento (%)')
ax.set_xlabel('Já viu colegas sofrerem consequências por problemas mentais?')
ax.set_ylabel('Proporção (%)')
ax.legend(title='Buscou Tratamento?', frameon=False)
for c in ax.containers:
    ax.bar_label(c, fmt='%.1f%%', padding=3, fontsize=9)
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('data/processed/ato3_cultura_punitiva.png', bbox_inches='tight')
plt.show()


**INSIGHT:** Quando o funcionário testemunha colegas sofrerem consequências por revelar
problemas mentais, sua própria taxa de busca por tratamento cai. O medo aprendido é real.

**GANCHO:** *"Benefícios de saúde mental sem cultura de segurança psicológica são
marketing, não cuidado. O problema não é falta de recursos — é falta de confiança."*


---
## Ato 4 — O Gran Finale: O Modelo Preditivo de Machine Learning

**Premissa:** O sistema falha. As pessoas têm medo de pedir ajuda.
Então criamos um modelo que permite à empresa agir **preventivamente** —
identificando grupos de risco antes que o burnout aconteça.


### 4.1 — Definição do Problema

**Target:** `work_interfere` (binarizado)
- **Classe 1 (Alto Risco):** `Often` ou `Sometimes`
- **Classe 0 (Baixo Risco):** `Rarely` ou `Never`

**Features:** `remote_work`, `anonymity`, `leave`, `no_employees`, `supervisor`
(variáveis comportamentais e ambientais — sem dados médicos)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    f1_score, accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

FEATURES = ['remote_work', 'anonymity', 'leave', 'no_employees', 'supervisor']
TARGET   = 'work_interfere'

# Preparação
df_ml = df[FEATURES + [TARGET]].dropna().copy()
df_ml[TARGET] = df_ml[TARGET].map({'Often': 1, 'Sometimes': 1, 'Rarely': 0, 'Never': 0})
df_ml = df_ml.dropna(subset=[TARGET])

print(f"Dataset de ML: {df_ml.shape}")
print(f"Distribuição do target:\n{df_ml[TARGET].value_counts()}")
print(f"Proporção: {df_ml[TARGET].mean():.2%} de alto risco")


In [ ]:
# Encoding das features categóricas
encoders = {}
df_enc = df_ml.copy()
for col in FEATURES:
    le = LabelEncoder()
    df_enc[col] = le.fit_transform(df_enc[col].astype(str))
    encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")


In [ ]:
# Split treino/teste (80/20, stratificado)
X = df_enc[FEATURES]
y = df_enc[TARGET].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treino: {X_train.shape[0]} | Teste: {X_test.shape[0]}")
print(f"Target treino — alto risco: {y_train.mean():.2%}")
print(f"Target teste  — alto risco: {y_test.mean():.2%}")


In [ ]:
# Treinamento dos modelos
rf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced')
lr = LogisticRegression(random_state=42, max_iter=500, class_weight='balanced')

rf.fit(X_train, y_train)
lr.fit(X_train, y_train)

y_pred_rf = rf.predict(X_test)
y_pred_lr = lr.predict(X_test)

print("=== RANDOM FOREST ===")
print(f"F1-Score:  {f1_score(y_test, y_pred_rf):.3f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_rf):.3f}")
print()
print("=== LOGISTIC REGRESSION ===")
print(f"F1-Score:  {f1_score(y_test, y_pred_lr):.3f}")
print(f"Accuracy:  {accuracy_score(y_test, y_pred_lr):.3f}")


In [ ]:
# Relatório completo e confusion matrix
print("=== RANDOM FOREST — Relatório Completo ===")
print(classification_report(y_test, y_pred_rf, target_names=['Baixo Risco', 'Alto Risco']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, y_pred, titulo in zip(
    axes,
    [y_pred_rf, y_pred_lr],
    ['Random Forest', 'Logistic Regression']
):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Baixo Risco', 'Alto Risco'])
    disp.plot(ax=ax, colorbar=False, cmap='YlGn')
    f1 = f1_score(y_test, y_pred)
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{titulo}\nF1={f1:.3f} | Acc={acc:.3f}')
    ax.spines[['top', 'right', 'bottom', 'left']].set_visible(False)

plt.suptitle('Confusion Matrix — Comparação de Modelos', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('data/processed/ato4_confusion_matrix.png', bbox_inches='tight')
plt.show()


In [ ]:
# Importância das features (Random Forest)
importancias = pd.DataFrame({
    'Feature': FEATURES,
    'Importância': rf.feature_importances_
}).sort_values('Importância', ascending=False)

print("Importância das features:")
print(importancias.to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 4))
importancias_plot = importancias.sort_values('Importância')
ax.barh(importancias_plot['Feature'], importancias_plot['Importância'],
        color='#4db6ac', edgecolor='none')
for i, (val, name) in enumerate(zip(importancias_plot['Importância'],
                                     importancias_plot['Feature'])):
    ax.text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=10)
ax.set_title('Importância das Features — Random Forest')
ax.set_xlabel('Importância')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.savefig('data/processed/ato4_feature_importance.png', bbox_inches='tight')
plt.show()


### 4.2 — A Proposta de Valor

**Nossa análise provou que benefícios não funcionam sem cultura e sigilo.**

Portanto, criamos um modelo que analisa o *formato de trabalho e o ambiente*.
Se o modelo classificar que um grupo de funcionários sob certas condições tem alta
possibilidade de interferência no trabalho (`work_interfere = Often/Sometimes`),
a empresa não precisa esperar eles pedirem socorro.

**Ela atua preventivamente:**
- Mudando a cultura daquele setor
- Ajustando o trabalho remoto
- Reforçando as políticas de sigilo de forma anônima e proativa

As features mais importantes confirmam as conclusões dos Atos 2 e 3:
o sigilo (`anonymity`), o suporte do supervisor e a facilidade de acesso à licença
são os maiores preditores de interferência no trabalho.


---
## Seção 6 — Conclusões

### Síntese dos Achados

1. **Ato 1 — A Bagagem Invisível:** Funcionários com histórico familiar de doenças mentais
   chegam à empresa com maior predisposição, e preferem omitir isso na entrevista por medo.
   O RH contrata sem conhecer a realidade.

2. **Ato 2 — O Ecossistema Corporativo:** O trabalho remoto amplifica a interferência da
   saúde mental e reduz o suporte social. Grandes corporações têm processos de licença
   pouco comunicados.

3. **Ato 3 — A Cultura do Medo:** Sem sigilo garantido, a busca por tratamento cai drasticamente.
   O medo aprendido — de ver colegas punidos — silencia toda a equipe. Benefícios sem cultura
   de segurança psicológica são ineficazes.

4. **Ato 4 — O Modelo Preditivo:** Um modelo baseado exclusivamente em variáveis comportamentais
   e ambientais é capaz de identificar grupos de alto risco antes do burnout —
   permitindo ação preventiva sem invasão de privacidade.

### Recomendações para Empresas

- **Garantir sigilo formal** nas políticas de saúde mental (comunicar proativamente)
- **Criar estrutura de suporte para times remotos** (check-ins regulares, acesso a supervisores)
- **Comunicar claramente** o processo de licença médica em todas as faixas de tamanho
- **Implementar o modelo preditivo** por setor/time para antecipação de riscos
- **Eliminar cultura punitiva** através de treinamento de liderança e documentação de políticas
